# 03. GPT-2 Small, ViT-B/16, DiT-B/2

텐서 크기만 줄이고 각 기준 모델의 구조와 계산 경로를 유지한다.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")


## 1. GPT-2 Small


In [ ]:
class GPT2CausalSelfAttention(nn.Module):
    def __init__(self, dim=48, heads=12):
        super().__init__()

        assert dim % heads == 0

        self.heads = heads
        self.head_dim = dim // heads

        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, past=None):
        batch_size, query_length, dim = x.shape

        qkv = self.qkv(x)
        qkv = qkv.view(
            batch_size,
            query_length,
            3,
            self.heads,
            self.head_dim,
        )

        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)

        past_length = 0
        if past is not None:
            past_k = past[:, 0]
            past_v = past[:, 1]

            past_length = past_k.size(-2)

            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)

        present = torch.stack([k, v], dim=1)

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        key_length = k.size(-2)

        query_positions = (
            past_length
            + torch.arange(
                query_length,
                device=x.device,
            )
        )
        key_positions = torch.arange(
            key_length,
            device=x.device,
        )

        causal_mask = (
            key_positions[None, :]
            <= query_positions[:, None]
        )

        scores = scores.masked_fill(
            ~causal_mask[None, None, :, :],
            torch.finfo(scores.dtype).min,
        )

        attention = scores.softmax(dim=-1)
        hidden = attention @ v

        hidden = hidden.transpose(1, 2).contiguous()
        hidden = hidden.view(
            batch_size,
            query_length,
            dim,
        )

        return self.proj(hidden), present


class GPT2MLP(nn.Module):
    def __init__(self, dim=48):
        super().__init__()

        self.fc = nn.Linear(dim, 4 * dim)
        self.proj = nn.Linear(4 * dim, dim)

    def forward(self, x):
        x = self.fc(x)
        x = F.gelu(
            x,
            approximate="tanh",
        )

        return self.proj(x)


class GPT2Block(nn.Module):
    def __init__(self, dim=48, heads=12):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            dim,
            eps=1e-5,
        )
        self.attn = GPT2CausalSelfAttention(
            dim,
            heads,
        )

        self.norm2 = nn.LayerNorm(
            dim,
            eps=1e-5,
        )
        self.mlp = GPT2MLP(dim)

    def forward(self, x, past=None):
        attention_output, present = self.attn(
            self.norm1(x),
            past=past,
        )

        x = x + attention_output
        x = x + self.mlp(
            self.norm2(x)
        )

        return x, present


class SmallWidthGPT2(nn.Module):
    def __init__(
        self,
        vocab_size=128,
        max_length=32,
        dim=48,
        depth=12,
        heads=12,
    ):
        super().__init__()

        self.max_length = max_length

        self.token_embedding = nn.Embedding(
            vocab_size,
            dim,
        )
        self.position_embedding = nn.Embedding(
            max_length,
            dim,
        )

        self.blocks = nn.ModuleList(
            [
                GPT2Block(
                    dim=dim,
                    heads=heads,
                )
                for _ in range(depth)
            ]
        )

        self.final_norm = nn.LayerNorm(
            dim,
            eps=1e-5,
        )

        self.initialize_weights()

    def initialize_weights(self):
        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=0.02,
        )
        nn.init.normal_(
            self.position_embedding.weight,
            mean=0.0,
            std=0.01,
        )

        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=0.02,
                )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(
                    module.weight
                )
                nn.init.zeros_(
                    module.bias
                )

    def forward(
        self,
        token_ids,
        past=None,
        use_cache=False,
    ):
        _, sequence_length = token_ids.shape

        if past is None:
            past_length = 0
        else:
            past_length = past.size(-2)

        if (
            past_length
            + sequence_length
            > self.max_length
        ):
            raise ValueError(
                "past length + sequence length exceeds max_length"
            )

        positions = (
            past_length
            + torch.arange(
                sequence_length,
                device=token_ids.device,
            )
        )

        x = (
            self.token_embedding(token_ids)
            + self.position_embedding(
                positions
            )[None, :, :]
        )

        presents = []

        for layer_index, block in enumerate(
            self.blocks
        ):
            if past is None:
                layer_past = None
            else:
                layer_past = past[
                    :,
                    layer_index,
                ]

            x, present = block(
                x,
                past=layer_past,
            )
            presents.append(present)

        x = self.final_norm(x)

        logits = F.linear(
            x,
            self.token_embedding.weight,
        )

        if not use_cache:
            return logits

        present = torch.stack(
            presents,
            dim=1,
        )

        return logits, present


gpt = SmallWidthGPT2().to(device)

token_ids = torch.randint(
    0,
    128,
    (2, 6),
    device=device,
)

gpt_logits, gpt_cache = gpt(
    token_ids,
    use_cache=True,
)

assert gpt_logits.shape == (2, 6, 128)
assert gpt_cache.shape == (
    2,
    12,
    2,
    12,
    6,
    4,
)

gpt_loss = F.cross_entropy(
    gpt_logits[:, :-1].reshape(-1, 128),
    token_ids[:, 1:].reshape(-1),
)
gpt_loss.backward()

next_token = torch.randint(
    0,
    128,
    (2, 1),
    device=device,
)

next_logits, next_cache = gpt(
    next_token,
    past=gpt_cache.detach(),
    use_cache=True,
)

assert next_logits.shape == (2, 1, 128)
assert next_cache.size(-2) == 7


## 2. ViT-B/16


In [ ]:
def lecun_normal_(tensor):
    fan_in = nn.init._calculate_correct_fan(
        tensor,
        mode="fan_in",
    )

    standard_deviation = math.sqrt(
        1.0 / fan_in
    )
    truncated_standard_deviation = (
        standard_deviation
        / 0.87962566103423978
    )

    return nn.init.trunc_normal_(
        tensor,
        mean=0.0,
        std=truncated_standard_deviation,
        a=-2.0
        * truncated_standard_deviation,
        b=2.0
        * truncated_standard_deviation,
    )


class ViTSelfAttention(nn.Module):
    def __init__(self, dim=48, heads=12):
        super().__init__()

        assert dim % heads == 0

        self.heads = heads
        self.head_dim = dim // heads

        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)

    def reshape_heads(self, x):
        batch_size, sequence_length, _ = (
            x.shape
        )

        x = x.view(
            batch_size,
            sequence_length,
            self.heads,
            self.head_dim,
        )

        return x.transpose(1, 2)

    def forward(self, x):
        q = self.reshape_heads(
            self.query(x)
        )
        k = self.reshape_heads(
            self.key(x)
        )
        v = self.reshape_heads(
            self.value(x)
        )

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(
            self.head_dim
        )

        attention = scores.softmax(dim=-1)
        hidden = attention @ v

        hidden = hidden.transpose(
            1,
            2,
        ).contiguous()
        hidden = hidden.view(
            x.size(0),
            x.size(1),
            x.size(2),
        )

        return self.proj(hidden)


class ViTMLP(nn.Module):
    def __init__(self, dim=48):
        super().__init__()

        self.fc1 = nn.Linear(
            dim,
            4 * dim,
        )
        self.fc2 = nn.Linear(
            4 * dim,
            dim,
        )

        self.dropout1 = nn.Dropout(0.0)
        self.dropout2 = nn.Dropout(0.0)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(
            x,
            approximate="tanh",
        )

        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.dropout2(x)

        return x


class ViTBlock(nn.Module):
    def __init__(self, dim=48, heads=12):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            dim,
            eps=1e-6,
        )
        self.attn = ViTSelfAttention(
            dim,
            heads,
        )
        self.attn_dropout = nn.Dropout(0.0)

        self.norm2 = nn.LayerNorm(
            dim,
            eps=1e-6,
        )
        self.mlp = ViTMLP(dim)

    def forward(self, x):
        attention_output = self.attn(
            self.norm1(x)
        )
        attention_output = (
            self.attn_dropout(
                attention_output
            )
        )

        x = x + attention_output
        x = x + self.mlp(
            self.norm2(x)
        )

        return x


class SmallWidthViTB16(nn.Module):
    def __init__(
        self,
        image_size=32,
        dim=48,
        depth=12,
        heads=12,
        classes=10,
    ):
        super().__init__()

        patch_size = 16
        patch_count = (
            image_size // patch_size
        ) ** 2

        self.patch_size = patch_size

        self.patch_embedding = nn.Conv2d(
            in_channels=3,
            out_channels=dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

        self.cls_token = nn.Parameter(
            torch.zeros(
                1,
                1,
                dim,
            )
        )
        self.position_embedding = nn.Parameter(
            torch.empty(
                1,
                patch_count + 1,
                dim,
            )
        )

        self.input_dropout = nn.Dropout(0.0)

        self.blocks = nn.ModuleList(
            [
                ViTBlock(
                    dim=dim,
                    heads=heads,
                )
                for _ in range(depth)
            ]
        )

        self.final_norm = nn.LayerNorm(
            dim,
            eps=1e-6,
        )
        self.head = nn.Linear(
            dim,
            classes,
        )

        self.initialize_weights()

    def initialize_weights(self):
        lecun_normal_(
            self.patch_embedding.weight
        )
        nn.init.zeros_(
            self.patch_embedding.bias
        )

        nn.init.zeros_(
            self.cls_token
        )
        nn.init.normal_(
            self.position_embedding,
            mean=0.0,
            std=0.02,
        )

        for block in self.blocks:
            attention_layers = [
                block.attn.query,
                block.attn.key,
                block.attn.value,
                block.attn.proj,
            ]

            for linear in attention_layers:
                nn.init.xavier_uniform_(
                    linear.weight
                )
                nn.init.zeros_(
                    linear.bias
                )

            mlp_layers = [
                block.mlp.fc1,
                block.mlp.fc2,
            ]

            for linear in mlp_layers:
                nn.init.xavier_uniform_(
                    linear.weight
                )
                nn.init.normal_(
                    linear.bias,
                    mean=0.0,
                    std=1e-6,
                )

        for module in self.modules():
            if isinstance(module, nn.LayerNorm):
                nn.init.ones_(
                    module.weight
                )
                nn.init.zeros_(
                    module.bias
                )

        nn.init.zeros_(
            self.head.weight
        )
        nn.init.zeros_(
            self.head.bias
        )

    def forward(self, image):
        x = self.patch_embedding(image)
        x = x.flatten(2).transpose(1, 2)

        cls_token = self.cls_token.expand(
            image.size(0),
            -1,
            -1,
        )

        x = torch.cat(
            [
                cls_token,
                x,
            ],
            dim=1,
        )

        x = (
            x
            + self.position_embedding[
                :,
                : x.size(1),
            ]
        )
        x = self.input_dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        return self.head(
            x[:, 0]
        )


vit = SmallWidthViTB16().to(device)

images = torch.randn(
    2,
    3,
    32,
    32,
    device=device,
)
image_labels = torch.tensor(
    [1, 2],
    device=device,
)

vit_logits = vit(images)

assert vit_logits.shape == (2, 10)

vit_loss = F.cross_entropy(
    vit_logits,
    image_labels,
)
vit_loss.backward()

assert (
    vit.head.weight.grad.norm()
    > 0
)


## 3. DiT-B/2


In [ ]:
def sincos_1d(dim, positions):
    assert dim % 2 == 0

    omega = torch.arange(
        dim // 2,
        dtype=torch.float64,
        device=positions.device,
    )
    omega = omega / (dim / 2.0)
    omega = 1.0 / (10000 ** omega)

    positions = positions.reshape(
        -1
    ).double()

    angles = torch.outer(
        positions,
        omega,
    )

    embedding = torch.cat(
        [
            angles.sin(),
            angles.cos(),
        ],
        dim=1,
    )

    return embedding.float()


def sincos_2d(
    dim,
    grid_size,
    device,
):
    assert dim % 2 == 0

    grid_h = torch.arange(
        grid_size,
        dtype=torch.float32,
        device=device,
    )
    grid_w = torch.arange(
        grid_size,
        dtype=torch.float32,
        device=device,
    )

    grid_y, grid_x = torch.meshgrid(
        grid_h,
        grid_w,
        indexing="ij",
    )

    embedding_x = sincos_1d(
        dim // 2,
        grid_x,
    )
    embedding_y = sincos_1d(
        dim // 2,
        grid_y,
    )

    return torch.cat(
        [
            embedding_x,
            embedding_y,
        ],
        dim=1,
    )


class TimestepEmbedder(nn.Module):
    def __init__(
        self,
        hidden_size,
        frequency_embedding_size=256,
    ):
        super().__init__()

        self.frequency_embedding_size = (
            frequency_embedding_size
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                frequency_embedding_size,
                hidden_size,
            ),
            nn.SiLU(),
            nn.Linear(
                hidden_size,
                hidden_size,
            ),
        )

    @staticmethod
    def timestep_embedding(
        t,
        dim,
        max_period=10000,
    ):
        half = dim // 2

        frequencies = torch.exp(
            -math.log(max_period)
            * torch.arange(
                start=0,
                end=half,
                dtype=torch.float32,
                device=t.device,
            )
            / half
        )

        args = (
            t[:, None].float()
            * frequencies[None]
        )

        embedding = torch.cat(
            [
                torch.cos(args),
                torch.sin(args),
            ],
            dim=-1,
        )

        if dim % 2:
            embedding = torch.cat(
                [
                    embedding,
                    torch.zeros_like(
                        embedding[:, :1]
                    ),
                ],
                dim=-1,
            )

        return embedding

    def forward(self, t):
        t_frequency = (
            self.timestep_embedding(
                t,
                self.frequency_embedding_size,
            )
        )

        return self.mlp(
            t_frequency
        )


class LabelEmbedder(nn.Module):
    def __init__(
        self,
        num_classes,
        hidden_size,
        dropout_prob,
    ):
        super().__init__()

        use_cfg_embedding = (
            dropout_prob > 0
        )

        self.embedding_table = nn.Embedding(
            num_classes
            + int(use_cfg_embedding),
            hidden_size,
        )

        self.num_classes = num_classes
        self.dropout_prob = dropout_prob

    def token_drop(
        self,
        labels,
        force_drop_ids=None,
    ):
        if force_drop_ids is None:
            drop_ids = (
                torch.rand(
                    labels.shape[0],
                    device=labels.device,
                )
                < self.dropout_prob
            )
        else:
            drop_ids = (
                force_drop_ids == 1
            )

        labels = torch.where(
            drop_ids,
            self.num_classes,
            labels,
        )

        return labels

    def forward(
        self,
        labels,
        train,
        force_drop_ids=None,
    ):
        use_dropout = (
            self.dropout_prob > 0
        )

        if (
            (train and use_dropout)
            or force_drop_ids is not None
        ):
            labels = self.token_drop(
                labels,
                force_drop_ids,
            )

        return self.embedding_table(
            labels
        )


def modulate(
    x,
    shift,
    scale,
):
    return (
        x
        * (1 + scale[:, None])
        + shift[:, None]
    )


class DiTSelfAttention(nn.Module):
    def __init__(self, dim=48, heads=12):
        super().__init__()

        assert dim % heads == 0

        self.heads = heads
        self.head_dim = dim // heads
        self.scale = (
            self.head_dim ** -0.5
        )

        self.qkv = nn.Linear(
            dim,
            3 * dim,
            bias=True,
        )
        self.proj = nn.Linear(
            dim,
            dim,
            bias=True,
        )

    def forward(self, x):
        batch_size, sequence_length, dim = (
            x.shape
        )

        qkv = self.qkv(x)
        qkv = qkv.view(
            batch_size,
            sequence_length,
            3,
            self.heads,
            self.head_dim,
        )

        q, k, v = qkv.permute(
            2,
            0,
            3,
            1,
            4,
        ).unbind(0)

        attention = (
            q @ k.transpose(-2, -1)
        ) * self.scale
        attention = attention.softmax(
            dim=-1
        )

        hidden = attention @ v
        hidden = hidden.transpose(
            1,
            2,
        ).contiguous()
        hidden = hidden.view(
            batch_size,
            sequence_length,
            dim,
        )

        return self.proj(hidden)


class DiTMLP(nn.Module):
    def __init__(self, dim=48):
        super().__init__()

        self.fc1 = nn.Linear(
            dim,
            4 * dim,
        )
        self.fc2 = nn.Linear(
            4 * dim,
            dim,
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(
            x,
            approximate="tanh",
        )

        return self.fc2(x)


class DiTBlock(nn.Module):
    def __init__(
        self,
        dim=48,
        heads=12,
    ):
        super().__init__()

        self.heads = heads

        self.norm1 = nn.LayerNorm(
            dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.attn = DiTSelfAttention(
            dim,
            heads,
        )

        self.norm2 = nn.LayerNorm(
            dim,
            elementwise_affine=False,
            eps=1e-6,
        )
        self.mlp = DiTMLP(dim)

        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                dim,
                6 * dim,
                bias=True,
            ),
        )

    def forward(
        self,
        x,
        condition,
    ):
        (
            shift_msa,
            scale_msa,
            gate_msa,
            shift_mlp,
            scale_mlp,
            gate_mlp,
        ) = self.adaLN_modulation(
            condition
        ).chunk(
            6,
            dim=1,
        )

        attention_input = modulate(
            self.norm1(x),
            shift_msa,
            scale_msa,
        )

        x = (
            x
            + gate_msa[:, None]
            * self.attn(
                attention_input
            )
        )

        mlp_input = modulate(
            self.norm2(x),
            shift_mlp,
            scale_mlp,
        )

        x = (
            x
            + gate_mlp[:, None]
            * self.mlp(
                mlp_input
            )
        )

        return x


class DiTFinalLayer(nn.Module):
    def __init__(
        self,
        dim,
        patch_size,
        out_channels,
    ):
        super().__init__()

        self.norm_final = nn.LayerNorm(
            dim,
            elementwise_affine=False,
            eps=1e-6,
        )

        self.linear = nn.Linear(
            dim,
            patch_size
            * patch_size
            * out_channels,
            bias=True,
        )

        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                dim,
                2 * dim,
                bias=True,
            ),
        )

    def forward(
        self,
        x,
        condition,
    ):
        shift, scale = (
            self.adaLN_modulation(
                condition
            ).chunk(
                2,
                dim=1,
            )
        )

        x = modulate(
            self.norm_final(x),
            shift,
            scale,
        )

        return self.linear(x)


class SmallWidthDiTB2(nn.Module):
    def __init__(
        self,
        image_size=8,
        in_channels=4,
        dim=48,
        depth=12,
        heads=12,
        num_classes=10,
        class_dropout_prob=0.1,
        learn_sigma=True,
    ):
        super().__init__()

        patch_size = 2

        self.in_channels = in_channels
        self.out_channels = (
            in_channels * 2
            if learn_sigma
            else in_channels
        )
        self.patch_size = patch_size
        self.num_heads = heads
        self.learn_sigma = learn_sigma

        self.patch_embedding = nn.Conv2d(
            in_channels=in_channels,
            out_channels=dim,
            kernel_size=patch_size,
            stride=patch_size,
            bias=True,
        )

        grid_size = (
            image_size // patch_size
        )
        num_patches = (
            grid_size * grid_size
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                num_patches,
                dim,
            ),
            requires_grad=False,
        )

        self.timestep_embedder = (
            TimestepEmbedder(dim)
        )
        self.label_embedder = (
            LabelEmbedder(
                num_classes,
                dim,
                class_dropout_prob,
            )
        )

        self.blocks = nn.ModuleList(
            [
                DiTBlock(
                    dim=dim,
                    heads=heads,
                )
                for _ in range(depth)
            ]
        )

        self.final_layer = (
            DiTFinalLayer(
                dim=dim,
                patch_size=patch_size,
                out_channels=(
                    self.out_channels
                ),
            )
        )

        self.initialize_weights()

    def initialize_weights(self):
        def basic_init(module):
            if isinstance(
                module,
                nn.Linear,
            ):
                nn.init.xavier_uniform_(
                    module.weight
                )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

        self.apply(basic_init)

        grid_size = int(
            self.position_embedding.shape[1]
            ** 0.5
        )

        position_embedding = sincos_2d(
            self.position_embedding.shape[-1],
            grid_size,
            self.position_embedding.device,
        )

        self.position_embedding.data.copy_(
            position_embedding.unsqueeze(0)
        )

        patch_weight = (
            self.patch_embedding.weight.data
        )
        nn.init.xavier_uniform_(
            patch_weight.view(
                patch_weight.shape[0],
                -1,
            )
        )
        nn.init.zeros_(
            self.patch_embedding.bias
        )

        nn.init.normal_(
            self.label_embedder.embedding_table.weight,
            std=0.02,
        )

        nn.init.normal_(
            self.timestep_embedder.mlp[0].weight,
            std=0.02,
        )
        nn.init.normal_(
            self.timestep_embedder.mlp[2].weight,
            std=0.02,
        )

        for block in self.blocks:
            nn.init.zeros_(
                block.adaLN_modulation[-1].weight
            )
            nn.init.zeros_(
                block.adaLN_modulation[-1].bias
            )

        nn.init.zeros_(
            self.final_layer.adaLN_modulation[
                -1
            ].weight
        )
        nn.init.zeros_(
            self.final_layer.adaLN_modulation[
                -1
            ].bias
        )

        nn.init.zeros_(
            self.final_layer.linear.weight
        )
        nn.init.zeros_(
            self.final_layer.linear.bias
        )

    def unpatchify(self, x):
        batch_size, token_count, _ = (
            x.shape
        )

        channels = self.out_channels
        patch_size = self.patch_size

        height = int(
            token_count ** 0.5
        )
        width = height

        assert (
            height * width
            == token_count
        )

        x = x.reshape(
            batch_size,
            height,
            width,
            patch_size,
            patch_size,
            channels,
        )

        x = torch.einsum(
            "nhwpqc->nchpwq",
            x,
        )

        return x.reshape(
            batch_size,
            channels,
            height * patch_size,
            width * patch_size,
        )

    def forward(
        self,
        x,
        t,
        y,
    ):
        x = self.patch_embedding(x)
        x = x.flatten(2).transpose(
            1,
            2,
        )

        x = (
            x
            + self.position_embedding.to(
                device=x.device,
                dtype=x.dtype,
            )
        )

        t = self.timestep_embedder(t)
        y = self.label_embedder(
            y,
            self.training,
        )
        condition = t + y

        for block in self.blocks:
            x = block(
                x,
                condition,
            )

        x = self.final_layer(
            x,
            condition,
        )

        return self.unpatchify(x)

    def forward_with_cfg(
        self,
        x,
        t,
        y,
        cfg_scale,
    ):
        half = x[
            : len(x) // 2
        ]
        combined = torch.cat(
            [
                half,
                half,
            ],
            dim=0,
        )

        model_output = self.forward(
            combined,
            t,
            y,
        )

        epsilon = model_output[
            :,
            :3,
        ]
        rest = model_output[
            :,
            3:,
        ]

        (
            conditional_epsilon,
            unconditional_epsilon,
        ) = torch.split(
            epsilon,
            len(epsilon) // 2,
            dim=0,
        )

        guided_epsilon = (
            unconditional_epsilon
            + cfg_scale
            * (
                conditional_epsilon
                - unconditional_epsilon
            )
        )

        epsilon = torch.cat(
            [
                guided_epsilon,
                guided_epsilon,
            ],
            dim=0,
        )

        return torch.cat(
            [
                epsilon,
                rest,
            ],
            dim=1,
        )


dit = SmallWidthDiTB2().to(device)

latent = torch.randn(
    2,
    4,
    8,
    8,
    device=device,
)
timesteps = torch.tensor(
    [500, 100],
    device=device,
)
class_labels = torch.tensor(
    [3, 4],
    device=device,
)

dit_output = dit(
    latent,
    timesteps,
    class_labels,
)

assert dit_output.shape == (
    2,
    8,
    8,
    8,
)
assert isinstance(
    dit.position_embedding,
    nn.Parameter,
)
assert not (
    dit.position_embedding.requires_grad
)

dit_target = torch.randn_like(
    dit_output
)
dit_loss = F.mse_loss(
    dit_output,
    dit_target,
)
dit_loss.backward()

assert (
    dit.final_layer.linear.weight.grad.norm()
    > 0
)

forced_labels = (
    dit.label_embedder.token_drop(
        torch.tensor(
            [1, 2],
            device=device,
        ),
        force_drop_ids=torch.tensor(
            [1, 0],
            device=device,
        ),
    )
)

assert forced_labels.tolist() == [
    10,
    2,
]
